# ResNet Model

### ResNet50
ResNet50 contains 50 learnable layers, organized into identity and convolutional residual blocks.

In [5]:
# PyTorch equivalents of the TensorFlow/Keras imports
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init

# Keras‑style layers we need to replace
from torch.nn import Conv2d, BatchNorm2d, MaxPool2d, AdaptiveAvgPool2d, Flatten, Linear, Dropout

In [6]:
def identity_block(x, f, filters):
    """
    PyTorch version of the Keras identity_block.
    """
    F1, F2, F3 = filters

    # 1×1 conv (pointwise)
    x_shortcut = x

    x = Conv2d(in_channels=x.shape[1],
               out_channels=F1,
               kernel_size=1,
               stride=1,
               bias=False)(x)
    x = BatchNorm2d(F1)(x)
    x = nn.ReLU(inplace=True)(x)

    # 3×3 conv
    x = Conv2d(in_channels=F1,
               out_channels=F2,
               kernel_size=f,
               stride=1,
               padding=f // 2,
               bias=False)(x)
    x = BatchNorm2d(F2)(x)
    x = nn.ReLU(inplace=True)(x)

    # 1×1 conv (final)
    x = Conv2d(in_channels=F2,
               out_channels=F3,
               kernel_size=1,
               stride=1,
               bias=False)(x)
    x = BatchNorm2d(F3)(x)

    # shortcut connection (add)
    shortcut = Conv2d(in_channels=x.shape[1],
                      out_channels=F3,
                      kernel_size=1,
                      stride=1,
                      bias=False)(x_shortcut)
    shortcut = BatchNorm2d(F3)(shortcut)

    x = Add.apply(x, shortcut)          # PyTorch does not have a built‑in Add layer
    x = nn.ReLU(inplace=True)(x)

    return x

In [7]:
def convolutional_block(x, f, filters, s=2):
    """
    PyTorch version of the Keras convolutional_block.
    """
    F1, F2, F3 = filters
    x_shortcut = x

    # 1×1 conv with stride `s`
    x = Conv2d(in_channels=x.shape[1],
               out_channels=F1,
               kernel_size=1,
               stride=s,
               bias=False)(x)
    x = BatchNorm2d(F1)(x)
    x = nn.ReLU(inplace=True)(x)

    # 3×3 conv
    x = Conv2d(in_channels=F1,
               out_channels=F2,
               kernel_size=f,
               stride=1,
               padding=f // 2,
               bias=False)(x)
    x = BatchNorm2d(F2)(x)
    x = nn.ReLU(inplace=True)(x)

    # 1×1 conv (final)
    x = Conv2d(in_channels=F2,
               out_channels=F3,
               kernel_size=1,
               stride=1,
               bias=False)(x)
    x = BatchNorm2d(F3)(x)

    # shortcut connection (1×1 conv on the shortcut path)
    shortcut = Conv2d(in_channels=x.shape[1],
                      out_channels=F3,
                      kernel_size=1,
                      stride=s,
                      bias=False)(x_shortcut)
    shortcut = BatchNorm2d(F3)(shortcut)

    x = x + shortcut
    x = nn.ReLU(inplace=True)(x)

    return x

In [8]:
class ResNet50(nn.Module):
    """
    PyTorch implementation of the scratch‑from‑scratch ResNet‑50
    architecture described in the original notebook.
    """
    def __init__(self, input_shape=(64, 64, 3), num_classes=10, training=False):
        super().__init__()

        # ---- Initial layers -------------------------------------------------
        self.pad = nn.ZeroPad2d(3)                         # ZeroPadding2D((3, 3))
        self.conv1 = Conv2d(3, 64, kernel_size=7, stride=2, bias=False)
        self.bn1   = BatchNorm2d(64)
        self.relu1 = nn.ReLU(inplace=True)
        self.maxpool = MaxPool2d(kernel_size=3, stride=2)

        # ---- Residual stages -----------------------------------------------
        # Stage 2 (64‑64‑256, stride 1)
        self.layer2 = self._make_layer(f=3, filters=[64, 64, 256], s=1)
        # Stage 3 (128‑128‑512, stride 2) – 3 blocks
        self.layer3 = self._make_layer(f=3, filters=[128, 128, 512], s=2)
        for _ in range(2):  # three identity blocks total in the original notebook
            self.layer3.add_module(f"id_{len(self.layer3)}", 
                                   self.identity_block(x=self.layer3[-1], f=3, filters=[128, 128, 512]))
        # Stage 4 (256‑256‑1024, stride 2) – 5 blocks
        self.layer4 = self._make_layer(f=3, filters=[256, 256, 1024], s=2)
        for _ in range(4):  # five identity blocks total
            self.layer4.add_module(f"id_{len(self.layer4)}", 
                                   self.identity_block(x=self.layer4[-1], f=3, filters=[256, 256, 1024]))
        # Stage 5 (512‑512‑2048, stride 2) – 3 blocks
        self.layer5 = self._make_layer(f=3, filters=[512, 512, 2048], s=2)
        for _ in range(2):  # three identity blocks total
            self.layer5.add_module(f"id_{len(self.layer5)}", 
                                   self.identity_block(x=self.layer5[-1], f=3, filters=[512, 512, 2048]))

        # ---- Classification head -------------------------------------------
        self.avgpool = AdaptiveAvgPool2d((1, 1))
        self.flatten = Flatten()
        self.fc    = Linear(2048, num_classes,
                            bias=True)  # GlorotUniform init is default in PyTorch

    # -----------------------------------------------------------------------
    def _make_layer(self, f, filters, s):
        """
        Helper that builds a convolutional_block.
        Returns a Sequential container so it can be added to the model.
        """
        return nn.Sequential(
            convolutional_block(x=self.layer4[-1] if hasattr(self, "layer4") else self,
                               f=f,
                               filters=filters,
                               s=s)
        )

    # -----------------------------------------------------------------------
    def identity_block(self, x, f, filters):
        """
        PyTorch version of the identity_block defined earlier.
        """
        return identity_block(x, f, filters)

    # -----------------------------------------------------------------------
    def forward(self, x):
        # Initial conv‑stack
        x = self.pad(x)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.maxpool(x)

        # Residual stages
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        # Classification head
        x = self.avgpool(x)
        x = self.flatten(x)
        return self.fc(x)

In [11]:
# Instantiate the model (same hyper‑parameters as the original notebook)
model = ResNet50(input_shape=(64, 64, 3), num_classes=4, training=False)

# ---- Print a model “summary” -------------------------------------------------
# PyTorch does not have a built‑in `summary` like Keras, but we can print
# the number of parameters per layer to get a comparable overview.
def print_model_summary(model):
    total_params = 0
    for name, module in model.named_children():
        param_cnt = sum(p.numel() for p in module.parameters())
        printable = f"{name:30s} : {param_cnt:,} params"
        print(printable)
        total_params += param_cnt
    print(f"\nTotal trainable parameters: {total_params:,}")

print_model_summary(model)

AttributeError: 'ResNet50' object has no attribute 'shape'

In [ ]:
# If you prefer the exact Keras‑style line‑by‑line output:
for i, layer in enumerate(model.children()):
    param_cnt = sum(p.numel() for p in layer.parameters())
    print(f"{i:2d} {layer.__class__.__name__:25s} {param_cnt:,} params")

0 input_layer_3 0
1 zero_padding2d_3 0
2 conv2d_6 9472
3 batch_normalization_6 256
4 activation_5 0
5 max_pooling2d_2 0
6 conv2d_7 4160
7 batch_normalization_7 256
8 activation_6 0
9 conv2d_8 36928
10 batch_normalization_8 256
11 activation_7 0
12 conv2d_9 16640
13 conv2d_10 16640
14 batch_normalization_9 1024
15 batch_normalization_10 1024
16 add_1 0
17 activation_8 0
18 conv2d_11 16448
19 batch_normalization_11 256
20 activation_9 0
21 conv2d_12 36928
22 batch_normalization_12 256
23 activation_10 0
24 conv2d_13 16640
25 batch_normalization_13 1024
26 add_2 0
27 activation_11 0
28 conv2d_14 16448
29 batch_normalization_14 256
30 activation_12 0
31 conv2d_15 36928
32 batch_normalization_15 256
33 activation_13 0
34 conv2d_16 16640
35 batch_normalization_16 1024
36 add_3 0
37 activation_14 0
38 conv2d_17 32896
39 batch_normalization_17 512
40 activation_15 0
41 conv2d_18 147584
42 batch_normalization_18 512
43 activation_16 0
44 conv2d_19 66048
45 conv2d_20 131584
46 batch_normalizatio